# Cell Type Annotation for Xenium Human Lymph Node

Hierarchical marker-based annotation of the 10x Xenium human lymph node dataset for downstream ligand-receptor analysis.

## Dataset

| | |
|:--|:--|
| **Source** | [10x Genomics — Xenium Prime 5K Human Pan Tissue and Pathways Panel](https://www.10xgenomics.com/datasets/preview-data-xenium-prime-gene-expression) |
| **Tissue** | FFPE Reactive Lymph Node (Left Axilla), Avaden Biosciences |
| **Panel** | Xenium Prime 5K (4,624 genes) |
| **Cells** | 708,647 |

## Pipeline

1. **Load** preprocessed Xenium h5ad (normalize_total + log1p)
2. **Lineage assignment** — score each cell against canonical marker panels (B, T, Myeloid, Stromal, NK)
3. **Subtype refinement** — deterministic marker rules within each lineage (first match wins)
4. **Validation** — marker dot plot, proportions, spatial co-localization
5. **Save** clean h5ad for CONSTELLATION

An appendix compares with CellTypist reference-based annotation and discusses its limitations on spatial data.

## Prerequisites

```bash
pip install scanpy celltypist pandas numpy matplotlib scipy
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def get_expression(adata, gene):
    """Get expression array for a gene."""
    if gene not in adata.var_names:
        return np.zeros(adata.n_obs)
    X = adata[:, gene].X
    if issparse(X):
        X = X.toarray().flatten()
    return np.asarray(X).flatten()

def get_expr_fraction(adata, mask, gene):
    """Get fraction of cells expressing a gene."""
    if gene not in adata.var_names:
        return 0.0
    X = adata[mask, gene].X
    if issparse(X):
        X = X.toarray()
    return float((X > 0).mean())

In [ ]:
# Paths (relative to lymph_node/code/)
DATA_PATH = '../data/xenium_lymphnode.h5ad'
OUTPUT_PATH = '../results/xenium_lymphnode_annotated.h5ad'

# Lineage markers for hierarchical gating
# Note: CXCL12 excluded from Stromal — it's a broadly expressed chemokine.
# CXCL12 is used later for FRC subtyping only.
LINEAGE_MARKERS = {
    'B_cell':  ['MS4A1', 'CD79A', 'CD79B', 'CD19', 'PAX5'],
    'T_cell':  ['CD3E', 'CD3D', 'CD3G', 'CD2', 'CD7'],
    'Myeloid': ['CD68', 'CD14', 'LYZ', 'CSF1R', 'ITGAM'],
    'Stromal': ['PECAM1', 'CDH5'],
    'NK':      ['NCAM1', 'NKG7', 'GNLY', 'KLRD1', 'KLRB1'],
}

LINEAGE_THRESHOLD = 0.2  # minimum score to assign a lineage

---
## 1. Load Data

The input h5ad was preprocessed from raw Xenium output with standard scanpy normalization. If starting from scratch, run the preprocessing below first.

<details>
<summary><b>Preprocessing from raw Xenium output</b> (click to expand)</summary>

Download the Xenium output bundle from [10x Genomics](https://www.10xgenomics.com/datasets/preview-data-xenium-prime-gene-expression) and extract to `../data/`. Then run:

```python
import scanpy as sc

# Load raw Xenium output
adata = sc.read_10x_h5('../data/cell_feature_matrix.h5')

# Add spatial coordinates from Xenium cell table
import pandas as pd
cells = pd.read_csv('../data/cells.csv.gz')
adata.obsm['spatial'] = cells[['x_centroid', 'y_centroid']].values

# Standard normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Compute neighborhood graph (used by CellTypist majority_voting)
sc.pp.highly_variable_genes(adata, flavor='seurat')
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)

adata.write_h5ad('../data/xenium_lymphnode.h5ad')
```

The key normalization is **normalize_total + log1p** (recorded in `adata.uns['log1p']`). This converts raw transcript counts to log-normalized expression values suitable for marker-based gating.

</details>

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
print(f'Cells: {adata.n_obs:,}')
print(f'Genes: {adata.n_vars:,}')
print(f'Spatial: {adata.obsm["spatial"].shape}')

# Drop any pre-existing annotation columns
for col in ['cell_type', 'cell_type_ingest', 'cell_type_hvg']:
    if col in adata.obs.columns:
        del adata.obs[col]

# Verify normalization status
# scanpy records log1p in adata.uns when sc.pp.log1p() is called
is_log1p = 'log1p' in adata.uns
if issparse(adata.X):
    sample = adata.X[:1000].toarray()
else:
    sample = np.asarray(adata.X[:1000])

print(f'\nNormalization:')
print(f'  adata.uns["log1p"]: {adata.uns.get("log1p", "NOT FOUND")}')
print(f'  Expression range: [{sample.min():.1f}, {sample.max():.1f}]')
print(f'  Sparsity: {(sample == 0).mean()*100:.1f}%')
if is_log1p:
    print(f'  Status: log-normalized (normalize_total + log1p)')
else:
    print(f'  WARNING: log1p key not found — data may be raw counts')

---
## 2. Hierarchical Lineage Gating

This is the core annotation step. Every cell is classified using canonical markers in two passes:

**Step 1 — Lineage assignment**: Score each cell against 5 lineage marker panels. Assign to the highest-scoring lineage above threshold (0.2). Cells below threshold are "Unassigned."

**Step 2 — Subtype refinement**: Within each lineage, apply deterministic marker rules in priority order (first match wins). The assignment order matters:

- **T cells**: T_CD8 → Tfh → T_naive → Treg → Tfr → T_helper → T_DN. CD8A first (high-confidence), FOXP3 later (spillover-prone).
- **B cells**: GC_DZ before B_activated (proliferating GC B cells can express XBP1). CD3E, CD68, and CD14 excluded from GC_DZ/GC_LZ to remove spatial transcript spillover.
- **Myeloid**: Macrophage (CD68+) → Monocyte (CD14+) → remaining Myeloid.
- **Stromal**: Endothelial (PECAM1+) → LEC (LYVE1+) → Pericyte (ACTA2+) → FRC (FAP+) → FDC (PDPN+) → Stromal.

Remaining cells fall to a catch-all type (B_mature, T_DN, etc.).

In [ ]:
# --- Hierarchical annotation diagram ---
fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlim(0, 15)
ax.set_ylim(4.5, 13)
ax.axis('off')

C = {
    'root': '#4a4a4a', 'B': '#6a5acd', 'T': '#cd7f32', 'M': '#3c7a5a',
    'S': '#c04040', 'NK': '#6aaa6a', 'U': '#888888',
}

def box(x, y, w, h, label, color, fontsize=8, bold=False):
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, linewidth=1.2,
                          edgecolor=color, facecolor=color, alpha=0.12,
                          zorder=2, clip_on=False)
    ax.add_patch(rect)
    rect2 = plt.Rectangle((x - w/2, y - h/2), w, h, linewidth=1.2,
                           edgecolor=color, facecolor='none', zorder=3, clip_on=False)
    ax.add_patch(rect2)
    ax.text(x, y, label, ha='center', va='center', fontsize=fontsize,
            fontweight='bold' if bold else 'normal', color=color, zorder=4, clip_on=False)

def conn(x1, y1, x2, y2, color='#aaaaaa', lw=0.8):
    ax.plot([x1, x2], [y1, y2], color=color, lw=lw, zorder=1, clip_on=False)

# ====== TITLE ======
ax.text(7.5, 12.6, 'Hierarchical Lineage Gating', ha='center', fontsize=15,
        fontweight='bold', color='#333', clip_on=False)

# ====== ROOT ======
box(7.5, 11.8, 3.2, 0.5, 'All Cells (708,647)', C['root'], fontsize=11, bold=True)

# ====== LINEAGE BOXES ======
lin_x = {'B': 1.5, 'T': 5.0, 'M': 8.5, 'S': 11.5, 'NK': 14.0}
lin_y = 10.5
for key, (lx, label, markers) in {
    'B':  (1.5,  'B_cell\n190,751',  'MS4A1 CD79A CD79B\nCD19 PAX5'),
    'T':  (5.0,  'T_cell\n301,338',  'CD3E CD3G CD2 CD7'),
    'M':  (8.5,  'Myeloid\n40,855',  'CD68 CD14 CSF1R\nITGAM'),
    'S':  (11.5, 'Stromal\n64,938',  'PECAM1 CDH5'),
    'NK': (14.0, 'NK\n13,067',       'NCAM1 KLRD1\nKLRB1'),
}.items():
    conn(7.5, 11.55, lx, lin_y + 0.35)
    box(lx, lin_y, 2.2, 0.6, label, C[key], fontsize=9, bold=True)
    ax.text(lx, lin_y - 0.5, markers, ha='center', va='top', fontsize=6,
            color='#777', style='italic', clip_on=False)

# ====== SUBTYPES (vertical lists under each lineage) ======
row_h = 0.42

def draw_subtypes(parent_x, parent_y, subtypes, color):
    start_y = parent_y - 1.3
    for i, (name, rule) in enumerate(subtypes):
        y = start_y - i * row_h
        box(parent_x - 0.15, y, 1.6, 0.33, name, color, fontsize=7.5, bold=True)
        ax.text(parent_x + 0.75, y, rule, ha='left', va='center', fontsize=6.5,
                color='#777', clip_on=False)
        if i == 0:
            conn(parent_x, parent_y - 0.3, parent_x - 0.15, y + 0.17, color=color)
        else:
            conn(parent_x - 0.15, y + row_h - 0.09, parent_x - 0.15, y + 0.17, color=color)

# B cell subtypes
b_subs = [
    ('PC',          'MZB1+ & XBP1>1 & MS4A1\u2013'),
    ('PB',          'MZB1+ & XBP1>1 & MS4A1+'),
    ('GC_DZ',       'MKI67+|TOP2A+ & CD3E\u2013 CD68\u2013 CD14\u2013'),
    ('B_activated', 'XBP1>1.5 & MZB1\u2013'),
    ('GC_LZ',       'BCL6+ & MKI67\u2013 & CD3E\u2013 CD68\u2013 CD14\u2013'),
    ('NBC',         'TCL1A+ | (IGHD+ IGHM+ CD27\u2013)'),
    ('MBC',         'CD27+'),
    ('B_mature',    'remaining B cells'),
]
draw_subtypes(lin_x['B'], lin_y, b_subs, C['B'])

# T cell subtypes
t_subs = [
    ('T_CD8',    'CD8A+'),
    ('Tfh',      'CD4+ & CXCR5+ & FOXP3\u2013'),
    ('T_naive',  'CCR7+ | LEF1+'),
    ('Treg',     'FOXP3+'),
    ('Tfr',      'CXCR5+ & FOXP3+ (from remaining)'),
    ('T_helper', 'CD4+'),
    ('T_DN',     'remaining T cells'),
]
draw_subtypes(lin_x['T'], lin_y, t_subs, C['T'])

# Myeloid subtypes
m_subs = [
    ('Macrophage', 'CD68+'),
    ('Monocyte',   'CD14+'),
    ('Myeloid',    'remaining'),
]
draw_subtypes(lin_x['M'], lin_y, m_subs, C['M'])

# Stromal subtypes
s_subs = [
    ('Endothelial', 'PECAM1+'),
    ('LEC',         'LYVE1+'),
    ('Pericyte',    'ACTA2+'),
    ('FRC',         'FAP+'),
    ('FDC',         'PDPN+'),
    ('Stromal',     'remaining'),
]
draw_subtypes(lin_x['S'], lin_y, s_subs, C['S'])

# NK — no subtypes
ax.text(lin_x['NK'], lin_y - 1.3, 'all NK lineage\ncells', ha='center', va='top',
        fontsize=6.5, color='#777', style='italic', clip_on=False)

# ====== FOOTER ======
ax.text(7.5, 4.8,
    'Assignment order matters: within each lineage, subtypes are checked top\u2192bottom (first match wins).\n'
    'GC_DZ and GC_LZ explicitly exclude CD3E+/CD68+/CD14+ cells to remove spatial spillover contamination.',
    ha='center', va='center', fontsize=7.5, color='#666', clip_on=False)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 1: Primary lineage assignment ---
print('Computing lineage scores...')
score_df = pd.DataFrame(index=adata.obs_names)

for lineage, markers in LINEAGE_MARKERS.items():
    avail = [m for m in markers if m in adata.var_names]
    if avail:
        detected = sum((get_expression(adata, m) > 0).astype(float) for m in avail)
        score_df[lineage] = detected / len(avail)
    else:
        score_df[lineage] = 0.0

# Assign to highest-scoring lineage above threshold
max_score = score_df.max(axis=1)
primary = score_df.idxmax(axis=1)
primary[max_score < LINEAGE_THRESHOLD] = 'Unassigned'
adata.obs['primary_lineage'] = primary.values

# Check marker availability
for lin, markers in LINEAGE_MARKERS.items():
    avail = [m for m in markers if m in adata.var_names]
    missing = [m for m in markers if m not in adata.var_names]
    print(f'  {lin}: {len(avail)}/{len(markers)} markers' +
          (f'  (missing: {missing})' if missing else ''))

# Distribution
print('\nLineage assignment:')
for lin, n in adata.obs['primary_lineage'].value_counts().items():
    print(f'  {lin:<12s}: {n:>8,} ({100*n/adata.n_obs:>5.1f}%)')

In [ ]:
# --- Step 2: Subtype refinement within lineages ---
cell_type = adata.obs['primary_lineage'].copy()

# Cache marker arrays
cd3e  = get_expression(adata, 'CD3E')
cd4   = get_expression(adata, 'CD4')
cd8a  = get_expression(adata, 'CD8A')
foxp3 = get_expression(adata, 'FOXP3')
cxcr5 = get_expression(adata, 'CXCR5')
ccr7  = get_expression(adata, 'CCR7')
lef1  = get_expression(adata, 'LEF1')

ms4a1 = get_expression(adata, 'MS4A1')
cd79a = get_expression(adata, 'CD79A')
bcl6  = get_expression(adata, 'BCL6')
mki67 = get_expression(adata, 'MKI67')
top2a = get_expression(adata, 'TOP2A') if 'TOP2A' in adata.var_names else np.zeros(adata.n_obs)
xbp1  = get_expression(adata, 'XBP1')
mzb1  = get_expression(adata, 'MZB1') if 'MZB1' in adata.var_names else np.zeros(adata.n_obs)
cd27  = get_expression(adata, 'CD27')
tcl1a = get_expression(adata, 'TCL1A') if 'TCL1A' in adata.var_names else np.zeros(adata.n_obs)
ighd  = get_expression(adata, 'IGHD') if 'IGHD' in adata.var_names else np.zeros(adata.n_obs)
ighm  = get_expression(adata, 'IGHM') if 'IGHM' in adata.var_names else np.zeros(adata.n_obs)

cd68   = get_expression(adata, 'CD68')
cd14   = get_expression(adata, 'CD14')

pecam1 = get_expression(adata, 'PECAM1')
lyve1  = get_expression(adata, 'LYVE1') if 'LYVE1' in adata.var_names else np.zeros(adata.n_obs)
acta2  = get_expression(adata, 'ACTA2') if 'ACTA2' in adata.var_names else np.zeros(adata.n_obs)
fap    = get_expression(adata, 'FAP') if 'FAP' in adata.var_names else np.zeros(adata.n_obs)
pdpn   = get_expression(adata, 'PDPN') if 'PDPN' in adata.var_names else np.zeros(adata.n_obs)

# ===== T CELLS =====
# Order: T_CD8 -> Tfh -> T_naive -> Treg -> Tfr -> T_helper -> T_DN
t_mask = cell_type == 'T_cell'
remaining_t = t_mask.copy()
print(f'T cells: {t_mask.sum():,}')

# T_CD8: CD8A+
t_cd8 = remaining_t & (cd8a > 0)
cell_type.loc[t_cd8] = 'T_CD8'
remaining_t &= ~t_cd8

# Tfh: CD4+ CXCR5+ FOXP3-
tfh = remaining_t & (cd4 > 0) & (cxcr5 > 0) & (foxp3 == 0)
cell_type.loc[tfh] = 'Tfh'
remaining_t &= ~tfh

# T_naive: CCR7+ or LEF1+
t_naive = remaining_t & ((ccr7 > 0) | (lef1 > 0))
cell_type.loc[t_naive] = 'T_naive'
remaining_t &= ~t_naive

# Treg: FOXP3+
treg = remaining_t & (foxp3 > 0)
cell_type.loc[treg] = 'Treg'
remaining_t &= ~treg

# Tfr: CXCR5+ FOXP3+ (from remaining — all FOXP3+ caught by Treg above)
tfr = remaining_t & (cxcr5 > 0) & (foxp3 > 0)
cell_type.loc[tfr] = 'Tfr'
remaining_t &= ~tfr

# T_helper: CD4+
t_helper = remaining_t & (cd4 > 0)
cell_type.loc[t_helper] = 'T_helper'
remaining_t &= ~t_helper

# T_DN: remaining T cells
cell_type.loc[remaining_t] = 'T_DN'

for st, n in cell_type[t_mask].value_counts().items():
    print(f'  {st}: {n:,} ({100*n/t_mask.sum():.1f}%)')

# ===== B CELLS =====
# Order: PC -> PB -> GC_DZ -> B_activated -> GC_LZ -> NBC -> MBC -> B_mature
b_mask = cell_type == 'B_cell'
remaining_b = b_mask.copy()
print(f'\nB cells: {b_mask.sum():,}')

# PC: MZB1+ XBP1>1.0 MS4A1-
pc = remaining_b & (mzb1 > 0) & (xbp1 > 1.0) & (ms4a1 == 0)
cell_type.loc[pc] = 'PC'
remaining_b &= ~pc

# PB: MZB1+ XBP1>1.0 MS4A1+
pb = remaining_b & (mzb1 > 0) & (xbp1 > 1.0) & (ms4a1 > 0)
cell_type.loc[pb] = 'PB'
remaining_b &= ~pb

# GC_DZ: proliferating (MKI67+ or TOP2A+), exclude CD3E/CD68/CD14 spillover
gc_dz = remaining_b & ((mki67 > 0) | (top2a > 0)) & (cd3e == 0) & (cd68 == 0) & (cd14 == 0)
cell_type.loc[gc_dz] = 'GC_DZ'
remaining_b &= ~gc_dz

# B_activated: XBP1>1.5 MZB1-
b_act = remaining_b & (xbp1 > 1.5) & (mzb1 == 0)
cell_type.loc[b_act] = 'B_activated'
remaining_b &= ~b_act

# GC_LZ: BCL6+ non-proliferating, exclude CD3E/CD68/CD14
gc_lz = remaining_b & (bcl6 > 0) & (mki67 == 0) & (top2a == 0) & (cd3e == 0) & (cd68 == 0) & (cd14 == 0)
cell_type.loc[gc_lz] = 'GC_LZ'
remaining_b &= ~gc_lz

# NBC: TCL1A+ or (IGHD+ IGHM+ CD27-)
nbc = remaining_b & ((tcl1a > 0) | ((ighd > 0) & (ighm > 0) & (cd27 == 0)))
cell_type.loc[nbc] = 'NBC'
remaining_b &= ~nbc

# MBC: CD27+
mbc = remaining_b & (cd27 > 0)
cell_type.loc[mbc] = 'MBC'
remaining_b &= ~mbc

# B_mature: remaining B cells
cell_type.loc[remaining_b] = 'B_mature'

for st, n in cell_type[b_mask].value_counts().items():
    print(f'  {st}: {n:,} ({100*n/b_mask.sum():.1f}%)')

# ===== MYELOID =====
# Macrophage -> Monocyte -> remaining Myeloid
m_mask = cell_type == 'Myeloid'
remaining_m = m_mask.copy()
print(f'\nMyeloid: {m_mask.sum():,}')

# Macrophage: CD68+
mac = remaining_m & (cd68 > 0)
cell_type.loc[mac] = 'Macrophage'
remaining_m &= ~mac

# Monocyte: CD14+
mono = remaining_m & (cd14 > 0)
cell_type.loc[mono] = 'Monocyte'
remaining_m &= ~mono

# Remaining stays as Myeloid

for st, n in cell_type[m_mask].value_counts().items():
    print(f'  {st}: {n:,} ({100*n/m_mask.sum():.1f}%)')

# ===== STROMAL =====
# Order: Endothelial -> LEC -> Pericyte -> FRC -> FDC -> Stromal
s_mask = cell_type == 'Stromal'
remaining_s = s_mask.copy()
print(f'\nStromal: {s_mask.sum():,}')

# Endothelial: PECAM1+
endo = remaining_s & (pecam1 > 0)
cell_type.loc[endo] = 'Endothelial'
remaining_s &= ~endo

# LEC: LYVE1+
lec = remaining_s & (lyve1 > 0)
cell_type.loc[lec] = 'LEC'
remaining_s &= ~lec

# Pericyte: ACTA2+
peri = remaining_s & (acta2 > 0)
cell_type.loc[peri] = 'Pericyte'
remaining_s &= ~peri

# FRC: FAP+
frc = remaining_s & (fap > 0)
cell_type.loc[frc] = 'FRC'
remaining_s &= ~frc

# FDC: PDPN+
fdc = remaining_s & (pdpn > 0)
cell_type.loc[fdc] = 'FDC'
remaining_s &= ~fdc

# Remaining stays as Stromal

for st, n in cell_type[s_mask].value_counts().items():
    print(f'  {st}: {n:,} ({100*n/s_mask.sum():.1f}%)')

# ===== NK =====
nk_mask = cell_type == 'NK'
print(f'\nNK: {nk_mask.sum():,}')

# Unassigned cells remain as-is (no recovery step)
n_unassigned = (cell_type == 'Unassigned').sum()
print(f'\nUnassigned: {n_unassigned:,} ({100*n_unassigned/adata.n_obs:.1f}%)')

adata.obs['cell_type'] = cell_type.values

In [ ]:
# Final distribution
ct = adata.obs['cell_type']
ct_counts = ct.value_counts()
n_assigned = (ct != 'Unassigned').sum()
n_unassigned = (ct == 'Unassigned').sum()
n_types = len([t for t in ct_counts.index if t != 'Unassigned'])

print(f'Cell types: {n_types}')
print(f'Assigned: {n_assigned:,} ({100*n_assigned/adata.n_obs:.1f}%)')
print(f'Unassigned: {n_unassigned:,} ({100*n_unassigned/adata.n_obs:.1f}%)')

print(f'\n{"Cell Type":<20s} {"Count":>10s} {"% Assigned":>12s}')
print('-' * 45)
for name, n in ct_counts.items():
    if name == 'Unassigned':
        continue
    print(f'{name:<20s} {n:>10,} {100*n/n_assigned:>10.1f}%')
print('-' * 45)
print(f'{"Total Assigned":<20s} {n_assigned:>10,}')
print(f'{"Unassigned":<20s} {n_unassigned:>10,}')

---
## 3. Validation

### Marker dot plot
Each cell type should express its canonical positive markers and lack negative markers. Dot size shows fraction of cells positive; color shows mean expression.

### Proportions & spatial co-localization
Compare to expected ranges for reactive lymph node, and verify spatial relationships.

In [ ]:
# --- Marker dot plot ---
# Order cell types by lineage
CT_ORDER = [
    'PC', 'PB', 'GC_DZ', 'B_activated', 'GC_LZ', 'NBC', 'MBC', 'B_mature',
    'T_CD8', 'Tfh', 'T_naive', 'Treg', 'T_helper', 'T_DN',
    'Macrophage', 'Monocyte', 'Myeloid',
    'Endothelial', 'FDC', 'FRC', 'Stromal',
    'NK',
]
# Filter to types with >0 cells
ct_present = [t for t in CT_ORDER if (adata.obs['cell_type'] == t).sum() > 0]

# Key markers grouped by lineage
DOTPLOT_MARKERS = {
    'B cell': ['MS4A1', 'CD79A', 'MZB1', 'XBP1', 'MKI67', 'BCL6', 'TCL1A', 'IGHD', 'CD27'],
    'T cell': ['CD3E', 'CD4', 'CD8A', 'FOXP3', 'CXCR5', 'CCR7', 'LEF1'],
    'Myeloid': ['CD68', 'CD14'],
    'Stromal/NK': ['PECAM1', 'PDPN', 'FAP', 'KLRB1'],
}

# Filter to genes in panel
dotplot_markers = {}
for group, genes in DOTPLOT_MARKERS.items():
    avail = [g for g in genes if g in adata.var_names]
    if avail:
        dotplot_markers[group] = avail

adata_plot = adata[adata.obs['cell_type'].isin(ct_present)].copy()

sc.pl.dotplot(adata_plot, var_names=dotplot_markers, groupby='cell_type',
              categories_order=ct_present, standard_scale='var',
              dendrogram=False, figsize=(10, 7), show=True)

del adata_plot

In [ ]:
# --- Proportion validation ---
EXPECTED = {
    'B cells':  (['B_mature','NBC','MBC','B_activated','GC_DZ','GC_LZ','PB','PC'], (0.25, 0.55)),
    'T cells':  (['T_naive','T_helper','T_CD8','T_DN','Treg','Tfh'],               (0.25, 0.55)),
    'NK':       (['NK'],                                                             (0.01, 0.10)),
    'Myeloid':  (['Macrophage','Monocyte','Myeloid'],                               (0.03, 0.15)),
    'Stromal':  (['Endothelial','LEC','Pericyte','FRC','FDC','Stromal'],            (0.05, 0.25)),
}

ct = adata.obs['cell_type']
n_assigned = (ct != 'Unassigned').sum()

print('Proportion validation:')
print(f'{"Lineage":<12s} {"Count":>10s} {"Observed":>10s} {"Expected":>15s} {"Status":>8s}')
print('-' * 60)
for lineage, (types, (lo, hi)) in EXPECTED.items():
    n = ct.isin(types).sum()
    pct = n / n_assigned
    status = 'PASS' if lo <= pct <= hi else 'WARN'
    print(f'{lineage:<12s} {n:>10,} {100*pct:>9.1f}% {100*lo:>5.0f}-{100*hi:<5.0f}% {status:>8s}')

n_cd4 = (ct == 'T_helper').sum()
n_cd8 = (ct == 'T_CD8').sum()
n_gc = ct.isin(['GC_DZ', 'GC_LZ']).sum()
n_b = ct.isin(EXPECTED['B cells'][0]).sum()
n_tfh = (ct == 'Tfh').sum()
n_t = ct.isin(EXPECTED['T cells'][0]).sum()
print(f'\nKey ratios:')
print(f'  CD4:CD8 = {n_cd4/n_cd8:.2f}:1 (expected 1.5-2.5:1)')
print(f'  GC/Total B = {100*n_gc/n_b:.1f}% (expected 5-15%)')
print(f'  Tfh/Total T = {100*n_tfh/n_t:.1f}% (expected 1-5%)')

# --- Spatial co-localization ---
print('\n--- Spatial Co-localization ---')
coords = adata.obsm['spatial']
tree = cKDTree(coords)
_, sp_idx = tree.query(coords, k=16)
sp_idx = sp_idx[:, 1:]

ct_arr = ct.values
COLOC = {
    ('GC_DZ', 'GC_LZ'):         'GC structure',
    ('GC_DZ', 'Tfh'):            'T-B help in GC',
    ('NBC', 'MBC'):              'B follicle',
    ('Endothelial', 'Pericyte'): 'Vasculature',
    ('T_naive', 'T_helper'):     'T zone',
    ('FDC', 'GC_LZ'):            'GC organization',
}

print(f'\n{"CT1":<15s} {"CT2":<15s} {"Enrichment":>12s} {"Biology":>20s}')
print('-' * 70)
for (ct1, ct2), biology in COLOC.items():
    m1 = ct_arr == ct1
    if m1.sum() == 0:
        continue
    expected = (ct_arr == ct2).sum() / len(ct_arr)
    observed = (ct_arr[sp_idx[m1].flatten()] == ct2).mean()
    enr = observed / expected if expected > 0 else 0
    print(f'{ct1:<15s} {ct2:<15s} {enr:>10.2f}x  {biology:>20s}')

---
## 4. Save

Save clean h5ad with `cell_type` and `primary_lineage`, ready for CONSTELLATION.

In [ ]:
adata_out = adata.copy()

# Keep only essential columns
adata_out.obs = adata_out.obs[['cell_type', 'primary_lineage']].copy()

# Keep only spatial in obsm
for key in list(adata_out.obsm.keys()):
    if key != 'spatial':
        del adata_out.obsm[key]

adata_out.write_h5ad(OUTPUT_PATH)

ct_counts = adata_out.obs['cell_type'].value_counts()
n_types = len([t for t in ct_counts.index if t != 'Unassigned'])
n_assigned = (adata_out.obs['cell_type'] != 'Unassigned').sum()

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {adata_out.shape}')
print(f'Cell types: {n_types}')
print(f'Assigned: {n_assigned:,} ({100*n_assigned/adata_out.n_obs:.1f}%)')
print(f'Unassigned: {adata_out.n_obs - n_assigned:,}')

---
## Appendix: CellTypist Reference-Based Annotation

[CellTypist](https://www.celltypist.org/) is a logistic regression classifier trained on scRNA-seq atlases. We run the **Human Tonsil** model (`Cells_Human_Tonsil.pkl`, King et al.) with `majority_voting=True` — which over-clusters the data (Leiden), then assigns each cluster the consensus type.

This section compares CellTypist output with our marker-based annotations and discusses why reference-based methods have systematic limitations on Xenium spatial data.

In [ ]:
import celltypist
from celltypist import models
import time

# --- Run CellTypist ---
model = models.Model.load(model='Cells_Human_Tonsil.pkl')
print(f'Model features: {len(model.features)}')
panel_overlap = set(model.features) & set(adata.var_names)
print(f'Overlap with Xenium panel: {len(panel_overlap)} / {len(model.features)} '
      f'({100*len(panel_overlap)/len(model.features):.0f}%)')

t0 = time.time()
predictions = celltypist.annotate(adata, model=model, majority_voting=True)
print(f'CellTypist completed in {time.time()-t0:.0f}s')

pred_adata = predictions.to_adata()
ct_labels = pred_adata.obs['majority_voting'].values
adata.obs['celltypist_label'] = ct_labels
del pred_adata

ct_counts = adata.obs['celltypist_label'].value_counts()
print(f'\nCellTypist types: {len(ct_counts)}')
print(f'Top 15:')
for t, n in ct_counts.head(15).items():
    print(f'  {t:<40s} {n:>7,} ({100*n/adata.n_obs:>4.1f}%)')

In [ ]:
# --- Compare CellTypist vs marker-based annotations ---

# Map CellTypist fine types to broad lineages
CT_TO_LINEAGE = {}
b_kw = ['NBC', 'MBC', 'DZ', 'LZ', 'GC', 'PC ', 'PB', 'preGC', 'T-Trans']
t_kw = ['Naive', 'CM ', 'Tfh', 'Treg', 'Eff-', 'Tfr', 'cycling', 'RM ', 'SCM ', 'T-Eff']
for t in ct_counts.index:
    if any(k in str(t) for k in b_kw):
        CT_TO_LINEAGE[t] = 'B_cell'
    elif any(k in str(t) for k in t_kw):
        CT_TO_LINEAGE[t] = 'T_cell'
    else:
        CT_TO_LINEAGE[t] = 'Other'

ct_lineage = adata.obs['celltypist_label'].map(CT_TO_LINEAGE).fillna('Other')
our_lineage = adata.obs['primary_lineage'].copy()
our_lineage[our_lineage == 'Unassigned'] = 'Other'
our_lineage[our_lineage == 'NK'] = 'Other'

# Lineage agreement
agree = (ct_lineage == our_lineage).mean()
print(f'Lineage-level agreement: {100*agree:.1f}%\n')

# Cross-tabulation
cross = pd.crosstab(our_lineage, ct_lineage, margins=True)
print('Lineage cross-tabulation (rows=marker-based, cols=CellTypist):')
print(cross.to_string())

# --- Key failure cases in CellTypist ---
print('\n--- CellTypist failure cases ---\n')

# 1. DZ misannotation: check myeloid/endothelial markers in CellTypist DZ cells
dz_types = [t for t in ct_counts.index if 'DZ' in str(t)]
for dz in dz_types:
    mask = adata.obs['celltypist_label'] == dz
    n = mask.sum()
    if n == 0:
        continue
    cd68_pct = get_expr_fraction(adata, mask, 'CD68') * 100
    pecam_pct = get_expr_fraction(adata, mask, 'PECAM1') * 100
    mki67_pct = get_expr_fraction(adata, mask, 'MKI67') * 100
    ms4a1_pct = get_expr_fraction(adata, mask, 'MS4A1') * 100
    print(f'  CellTypist "{dz}" (n={n:,}):')
    print(f'    MS4A1+: {ms4a1_pct:.0f}%, MKI67+: {mki67_pct:.0f}%, '
          f'CD68+: {cd68_pct:.0f}%, PECAM1+: {pecam_pct:.0f}%')

    # What did our method call these cells?
    our_types = adata.obs.loc[mask, 'cell_type'].value_counts().head(5)
    print(f'    Our annotation: {", ".join(f"{t} ({n})" for t, n in our_types.items())}')
    print()

# 2. Overall type fragmentation
print(f'CellTypist: {len(ct_counts)} types (many are tonsil-specific, e.g., "IFN-activated", "DZ late Sphase")')
our_counts = adata.obs['cell_type'].value_counts()
n_our = len([t for t in our_counts.index if t != 'Unassigned'])
print(f'Marker-based: {n_our} types (lymph node-appropriate)')

In [ ]:
# --- Visual comparison ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: CellTypist lineage proportions
ct_lin_counts = ct_lineage.value_counts()
ax = axes[0]
ax.pie(ct_lin_counts.values, labels=ct_lin_counts.index,
       colors=['#6a5acd', '#cd7f32', '#888888'], autopct='%1.1f%%', startangle=90)
ax.set_title('CellTypist Lineages', fontweight='bold')

# Panel 2: Marker-based lineage proportions
our_lin_counts = our_lineage.value_counts()
ax = axes[1]
colors = {'B_cell': '#6a5acd', 'T_cell': '#cd7f32', 'Myeloid': '#3c7a5a',
          'Stromal': '#c04040', 'Other': '#888888'}
ax.pie(our_lin_counts.values, labels=our_lin_counts.index,
       colors=[colors.get(l, '#888') for l in our_lin_counts.index],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Marker-Based Lineages', fontweight='bold')

# Panel 3: GC_DZ marker purity comparison
# CellTypist DZ cells vs our GC_DZ
ct_dz_mask = adata.obs['celltypist_label'].str.contains('DZ', na=False)
our_dz_mask = adata.obs['cell_type'] == 'GC_DZ'

markers_check = ['MS4A1', 'MKI67', 'CD3E', 'CD68', 'PECAM1']
ct_fracs = [get_expr_fraction(adata, ct_dz_mask, m) * 100 for m in markers_check]
our_fracs = [get_expr_fraction(adata, our_dz_mask, m) * 100 for m in markers_check]

ax = axes[2]
x = np.arange(len(markers_check))
w = 0.35
bars1 = ax.bar(x - w/2, ct_fracs, w, label=f'CellTypist DZ (n={ct_dz_mask.sum():,})',
               color='#aaaaaa', edgecolor='#666')
bars2 = ax.bar(x + w/2, our_fracs, w, label=f'Marker GC_DZ (n={our_dz_mask.sum():,})',
               color='#6a5acd', edgecolor='#444')
ax.set_xticks(x)
ax.set_xticklabels(markers_check)
ax.set_ylabel('% cells positive')
ax.set_title('GC_DZ Marker Purity', fontweight='bold')
ax.legend(fontsize=8)
ax.axhline(30, color='red', ls='--', lw=0.8, alpha=0.5)
ax.text(len(markers_check) - 0.5, 32, '30% threshold', fontsize=7, color='red', alpha=0.7)

plt.tight_layout()
plt.show()

print(f'\nCellTypist DZ cells: {ct_dz_mask.sum():,} — includes CD68+ macrophages and PECAM1+ endothelial')
print(f'Marker-based GC_DZ: {our_dz_mask.sum():,} — pure B cells (MKI67+, CD3E\u2013, CD68\u2013, CD14\u2013)')

### Limitations of reference-based annotation on Xenium

| Issue | Cause | Impact |
|:------|:------|:-------|
| **Low feature overlap** | Xenium panel has ~500 genes; CellTypist tonsil model uses 5,005 features → only 1,531 overlap (31%). | Classifier operates on a fraction of its trained feature space, reducing discriminative power. |
| **Transcript spillover** | In spatial data, transcripts from neighboring cells leak into adjacent segmentation masks. | B cells near T cells gain CD3E signal; GC_DZ cells near macrophages gain CD68. CellTypist cannot distinguish real co-expression from spatial artifact. |
| **No spatial context** | CellTypist treats each cell independently — no information about tissue architecture. | Cannot leverage the fact that GC_DZ should be in germinal centers, FRC in T zones, etc. Marker-based gating + spatial validation catches errors that cell-level classifiers miss. |
| **Tissue mismatch** | Model trained on tonsil (dissociated scRNA-seq) applied to lymph node (spatial FFPE). | Tonsil-specific subtypes (e.g., "DZ late Sphase", "IFN-activated") may not exist in lymph node. Cell states differ between dissociated and in situ measurements. |
| **DZ misannotation** | CellTypist's "DZ" categories include cells with myeloid markers (CD68+, CD14+) and endothelial markers (PECAM1+). | Without marker filtering, ~60% of CellTypist DZ cells are non-B contaminants — macrophages, endothelial, and other cell types that share partial transcriptomic profiles with proliferating GC B cells. |
| **Majority voting artifacts** | `majority_voting` uses over-clustering (Leiden), then assigns the most common per-cell label to each cluster. | Spatially adjacent but biologically distinct cells (e.g., Tfh and GC_DZ at the T-B interface) can merge into a single cluster, causing cross-lineage assignment. |

**Conclusion**: For Xenium spatial data, marker-based hierarchical gating produces cleaner annotations than reference-based transfer, particularly for specialized populations like GC_DZ that are critical for ligand-receptor analysis. CellTypist provides a useful starting point for exploratory analysis but should not be used as the sole annotation method.